# `00_validate.ipynb` — End-to-end validation of the contactless AF pipeline

**What this notebook does**

1. **Validate the ground truth (GT).** Detect R-peaks on the *reference* ECG with a robust,
   artifact-aware detector (neurokit2), build the RR series per window, and check *when / where*
   AF is present — then **measure the detector against the known clinical labels** so we can trust
   it instead of eyeballing. Includes overlay plots (ECG + detected peaks) as direct proof.
2. **Extract features** for the three contactless modalities (cECG, PPG, BCG); BCG uses the
   paper-faithful **SWT+PAPR** morphology features (no RR/HRV — there is no reliable BCG peak
   detector, which is exactly why RR is dropped for BCG).
3. **Train three experts** (cECG; PPG = ppg1+ppg2; BCG = bcg1+bcg2), each on its own signals,
   under **Leave-One-Patient-Out (LOPO)**.
4. **SQI gating network** — a small NN that reads per-window signal-quality (SQI) and outputs 3
   weights to blend the experts.
5. **Results** — per-expert + gated metrics, **plus a per-window inspector** (expert guesses and
   gate weights for any single window) and a **diagnostics** section explaining *why* the gate
   does or doesn't help, with concrete improvement levers.

The reference ECG is used **only** to make/validate labels — never as an expert feature (at
deployment there is no reference ECG; the point is *contactless* detection).

> **All knobs live in Section 0.** A consolidated "what to try" list is in the final section.
> **Future:** the `GatingNet` is written to be swapped for a PyTorch module + Optuna once the
> design is shown to beat the naive mean here.

## 0 · Control panel — all tunable settings

> **One-time install for the robust detector:** `pip install neurokit2`
> (the notebook falls back to a basic built-in detector if it is missing, but that detector
> over-/under-counts beats on noisy windows and produces false AF — see Section 2).

In [17]:
from pathlib import Path

import sys, os

# Set working directory to your project root
os.chdir('/home/nik/projects/BA')

# ── PATHS ──────────────────────────────────────────────────────────────────
# Each patient = a subfolder PAT* under DATA_ROOT containing:
#   <PATxxx>/Signals/New_Data.csv      (contactless signals, fs = FS)
#   <PATxxx>/gt/NOM_ECG_ELEC_POTL_IIWaveExport.csv   (reference ECG, GT_FS)
#   <PATxxx>/alignment.json            (offsets + polarity flips)
REPO_SRC    = "src/"
DATA_ROOT   = "data/patients"                  # <-- EDIT ME (folder of PAT* dirs)
AF_LIST     = "data/AF_patients.txt"           # <-- EDIT ME (one AF patient-id per line)
RESULTS_DIR = "results/00"

# ── SAMPLING ────────────────────────────────────────────────────────────────
FS, GT_FS = 128, 500                            # contactless / reference-ECG sample rates [Hz]

# ── WINDOWING ────────────────────────────────────────────────────────────────
WINDOW_SEC = 30      # TWEAK (10-30). Window for GT check AND features. Paper uses 30 s.
HOP_SEC    = 15      # feature-grid step (overlap). GT timeline uses non-overlapping windows.

# ── GT R-PEAK DETECTOR (Section 2) ───────────────────────────────────────────
RPEAK_METHOD   = "neurokit"   # neurokit2 method: 'neurokit' | 'pantompkins1985' | 'hamilton2002'
# window quality gate: a window with an implausible HR or too few beats is marked n/a (not AF),
# so bad detection can NEVER masquerade as AF irregularity.
GT_MIN_BEATS   = 6            # need >= this many beats in a window to judge it
GT_HR_RANGE    = (30, 220)    # plausible mean HR [bpm]; outside -> n/a

# ── RR -> AF RULE (Section 2) ────────────────────────────────────────────────
AF_CV_THR  = 0.10    # coefficient of variation of RR (std/mean). Higher = stricter.
AF_PNN_THR = 0.30    # fraction of |dRR| > 50 ms. Higher = stricter.
RR_PHYS    = (0.3, 2.0)   # keep RR in [0.3, 2.0] s before stats

# ── LABEL SOURCE ──────────────────────────────────────────────────────────────
# 'patient'  : label every window by AF_LIST membership (your case: 21 AF + 19 control patients).
# 'gt_window': label each window by the Section-2 GT detector (use only if ALL patients are AF).
LABEL_SOURCE = "patient"

# ── BCG FEATURE PATH ─────────────────────────────────────────────────────────
BCG_MODE = "wavelet"          # 'wavelet' = SWT+PAPR (Yu et al.); 'rr' = legacy A/B path

# ── MODELS / CV ──────────────────────────────────────────────────────────────
RANDOM_STATE = 0

# ── GATING NET ────────────────────────────────────────────────────────────────
GATE_HIDDEN   = 8
GATE_EPOCHS   = 400
GATE_LR       = 0.05
GATE_TEMP     = 1.0      # softmax temperature (>1 = softer/closer to equal weights)
GATE_WD       = 1e-3     # L2 weight decay; pulls weights toward uniform -> graceful fallback to mean
GATE_CALIBRATE = False   # isotonic-calibrate expert probs before blending (diagnostic, Section 8)

sys.path.append(REPO_SRC)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print("Config:", dict(LABEL_SOURCE=LABEL_SOURCE, WINDOW_SEC=WINDOW_SEC, BCG_MODE=BCG_MODE,
                      RPEAK_METHOD=RPEAK_METHOD))

Config: {'LABEL_SOURCE': 'patient', 'WINDOW_SEC': 30, 'BCG_MODE': 'wavelet', 'RPEAK_METHOD': 'neurokit'}


## 1 · Imports & patient discovery

In [18]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import find_peaks
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, confusion_matrix

from signal_loader import PatientSignals
import extract as X
import features as F

try:
    import neurokit2 as nk
    HAVE_NK = True
    print("neurokit2", nk.__version__, "available -> robust R-peak detection ON")
except ImportError:
    HAVE_NK = False
    print("WARNING: neurokit2 not installed -> falling back to basic detector.\n"
          "         Run `pip install neurokit2` for trustworthy GT validation (see Section 2).")

af_set = set()
try:
    af_set = {l.strip() for l in open(AF_LIST) if l.strip()}
except FileNotFoundError:
    print("WARNING: AF_LIST not found — 'patient' label mode will see no positives.")

patients = sorted(d.name for d in Path(DATA_ROOT).iterdir()
                  if d.is_dir() and d.name.startswith("PAT")) if Path(DATA_ROOT).exists() else []
print(f"{len(patients)} patients found; {len(af_set & set(patients))} listed as AF.")

neurokit2 0.2.13 available -> robust R-peak detection ON
40 patients found; 19 listed as AF.


## 2 · Validate the ground truth — *robustly*

**Why the previous detector failed.** A homemade derivative→square→threshold detector is fragile:
on a noisy 30 s sinus window it finds far too many peaks (spurious) and on a motion/amplitude
artifact it finds too few. Either way the RR series looks irregular → the window is wrongly called
**AF**. That is exactly the pattern of false-AF on non-AF patients you saw.

**Fix:** detect R-peaks with **neurokit2** (adaptive thresholding + artifact correction), run **once
over the whole recording** (more context = better thresholds), then slice the RR series per window.
A per-window **quality gate** marks windows with implausible HR / too few beats as *n/a* instead of
forcing an AF decision, so bad detection can never masquerade as AF.

We then **measure the detector against your clinical labels** (AF list) rather than trusting it
blindly: AF patients should be mostly AF-windows, controls mostly non-AF.

In [19]:
def detect_rpeaks(ecg_filt, fs, method=RPEAK_METHOD):
    """R-peak sample indices over a full recording. neurokit2 if available, else basic fallback."""
    x = np.asarray(ecg_filt, float)
    x = np.where(np.isfinite(x), x, np.nanmedian(x[np.isfinite(x)]) if np.isfinite(x).any() else 0.0)
    if HAVE_NK:
        try:
            clean = nk.ecg_clean(x, sampling_rate=fs, method=method)
            _, info = nk.ecg_peaks(clean, sampling_rate=fs, method=method, correct_artifacts=True)
            return np.asarray(info["ECG_R_Peaks"], int)
        except Exception as e:
            print("neurokit failed, fallback:", e)
    # ---- basic fallback (Pan–Tompkins-lite) ----
    d = np.diff(x, prepend=x[0]); sq = d * d
    w = max(1, int(round(0.150 * fs)))
    integ = np.convolve(sq, np.ones(w) / w, mode="same")
    thr = 0.30 * (np.mean(integ) + np.std(integ))
    pk, _ = find_peaks(integ, height=thr, distance=max(1, int(round(0.20 * fs))))
    return pk


def rr_af_metrics(rr):
    """AF-relevant RR-irregularity metrics for one window."""
    drr = np.diff(rr)
    h, _ = np.histogram(drr, bins=16, range=(-0.4, 0.4))
    p = h / max(h.sum(), 1); p = p[p > 0]
    return dict(n_rr=len(rr), mean_hr=60.0 / np.mean(rr),
                rmssd=float(np.sqrt(np.mean(drr ** 2))),
                cv=float(np.std(rr) / np.mean(rr)),
                pnn=float(np.mean(np.abs(drr) > 0.05)),
                shannon=float(-np.sum(p * np.log(p)) / np.log(16)))


def classify_af(m, cv_thr=AF_CV_THR, pnn_thr=AF_PNN_THR):
    if m is None: return np.nan                       # undecidable window
    return int((m["cv"] >= cv_thr) or (m["pnn"] >= pnn_thr))


def window_metrics(rpeaks_full, fs, win_start, win_len):
    """RR metrics for the window [win_start, win_start+win_len] samples, with a quality gate."""
    pk = rpeaks_full[(rpeaks_full >= win_start) & (rpeaks_full < win_start + win_len)]
    if len(pk) < GT_MIN_BEATS: return None
    rr = np.diff(pk) / fs
    rr = rr[(rr > RR_PHYS[0]) & (rr < RR_PHYS[1])]
    if len(rr) < GT_MIN_BEATS - 1: return None
    m = rr_af_metrics(rr)
    if not (GT_HR_RANGE[0] <= m["mean_hr"] <= GT_HR_RANGE[1]): return None   # implausible -> n/a
    return m


def gt_window_table(gt_filt, gt_fs, window_s):
    """Non-overlapping per-window GT table for one patient (detection done once on full record)."""
    rpk = detect_rpeaks(gt_filt, gt_fs)
    n, wlen = len(gt_filt), int(window_s * gt_fs)
    rows = []
    for k, start in enumerate(range(0, max(1, n - wlen + 1), wlen)):
        m = window_metrics(rpk, gt_fs, start, wlen)
        rows.append(dict(win_idx=k, t_start_s=start / gt_fs, label=classify_af(m),
                         **(m or dict(n_rr=0, mean_hr=np.nan, rmssd=np.nan,
                                      cv=np.nan, pnn=np.nan, shannon=np.nan))))
    return pd.DataFrame(rows), rpk

In [ ]:
# Run GT validation for every patient.
gt_tables, gt_rpeaks, gt_signals, summary = {}, {}, {}, []
for pid in patients:
    try:
        pat = PatientSignals(str(Path(DATA_ROOT) / pid))
        pat.filter_all(FS); pat.offset_correction()      # GT gets only alignment.json flips
        if pat.gt_ecg_filt is None:
            print(f"{pid}: no GT ECG, skipped"); continue
        g = np.asarray(pat.gt_ecg_filt, float)
        t, rpk = gt_window_table(g, GT_FS, WINDOW_SEC)
        gt_tables[pid], gt_rpeaks[pid], gt_signals[pid] = t, rpk, g
        dec = t["label"].dropna()
        summary.append(dict(patient=pid, clinical=("AF" if pid in af_set else "non-AF"),
                            n_win=len(t), n_decided=int(dec.size),
                            af_frac=float((dec == 1).mean()) if dec.size else np.nan,
                            median_hr=float(np.nanmedian(t["mean_hr"]))))
    except Exception as e:
        print(f"{pid}: GT validation failed -> {e}")

gt_summary = pd.DataFrame(summary)
if not gt_summary.empty:
    display(gt_summary.sort_values(["clinical", "af_frac"], ascending=[True, False])
                      .round(3).reset_index(drop=True))

offset_correction: loaded alignment.json  [supersedes all]
  flips          = {'cecg': False, 'ppg1': False, 'ppg2': False, 'bcg1': False, 'bcg2': False}
  gt_flipped     = False
  initial_offset = 195  final_offset = 90
  trim_start_s   = 0.00  trim_end_s = 0.00


In [ ]:
# ── QA #1: does the detector's AF-fraction separate AF from control patients? ──
if not gt_summary.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for i, grp in enumerate(["AF", "non-AF"]):
        vals = gt_summary.loc[gt_summary.clinical == grp, "af_frac"].dropna()
        ax[0].scatter(np.full(len(vals), i) + np.random.uniform(-.08, .08, len(vals)),
                      vals, alpha=.7, label=grp)
    ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["AF (clinical)", "non-AF (clinical)"])
    ax[0].set_ylabel("fraction of windows called AF"); ax[0].axhline(0.5, ls="--", c="grey")
    ax[0].set_title("Detector AF-fraction vs clinical label\n(good = AF high, non-AF low)")

    # window-level detector quality vs clinical labels (treat patient label as window truth)
    y_true, y_pred = [], []
    for _, r in gt_summary.iterrows():
        t = gt_tables[r.patient]["label"].dropna()
        y_true += [int(r.clinical == "AF")] * len(t); y_pred += list(t.astype(int))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"GT detector vs clinical labels (window-level): "
          f"sens={tp/(tp+fn):.3f}  spec={tn/(tn+fp):.3f}  "
          f"({tp+tn}/{len(y_true)} windows agree)")
    print("Mismatched patients (AF<0.5 although clinical AF, or AF>0.5 although control):")
    bad = gt_summary[((gt_summary.clinical == "AF") & (gt_summary.af_frac < 0.5)) |
                     ((gt_summary.clinical == "non-AF") & (gt_summary.af_frac > 0.5))]
    display(bad.round(3))
    ax[1].hist(gt_summary.loc[gt_summary.clinical=="AF","af_frac"].dropna(), bins=10, alpha=.6, label="AF")
    ax[1].hist(gt_summary.loc[gt_summary.clinical=="non-AF","af_frac"].dropna(), bins=10, alpha=.6, label="non-AF")
    ax[1].set_xlabel("AF-window fraction"); ax[1].set_title("Separation histogram"); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# ── QA #2: VISUAL proof — overlay ECG + detected R-peaks for a few windows ─────
def plot_overlay(pids, title):
    pids = [p for p in pids if p in gt_signals][:2]
    if not pids: return
    fig, axes = plt.subplots(len(pids), 1, figsize=(12, 2.4 * len(pids)), squeeze=False)
    for ax, pid in zip(axes[:, 0], pids):
        g, rpk = gt_signals[pid], gt_rpeaks[pid]
        seg = slice(0, int(10 * GT_FS))                      # first 10 s
        ax.plot(np.arange(seg.stop) / GT_FS, g[seg], lw=.7)
        inwin = rpk[rpk < seg.stop]
        ax.plot(inwin / GT_FS, g[inwin], "rv", ms=6)
        ax.set_title(f"{pid} ({'AF' if pid in af_set else 'non-AF'}) — "
                     f"{len(inwin)} beats in 10 s ≈ {len(inwin)*6} bpm")
        ax.set_xlabel("s")
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_overlay([p for p in patients if p in af_set], "Detected R-peaks — AF patients")
plot_overlay([p for p in patients if p not in af_set], "Detected R-peaks — control patients")

In [ ]:
# ── per-patient AF timeline (green=AF, red=non-AF, grey=n/a) ───────────────────
if gt_tables:
    pids = list(gt_tables)
    fig, ax = plt.subplots(figsize=(11, 0.32 * len(pids) + 1))
    cmap = {1: "#2e7d32", 0: "#c62828"}
    for row, pid in enumerate(pids):
        for _, r in gt_tables[pid].iterrows():
            ax.barh(row, WINDOW_SEC, left=r["t_start_s"], height=0.8,
                    color=cmap.get(r["label"], "#bdbdbd"))
    ax.set_yticks(range(len(pids)))
    ax.set_yticklabels([f"{p}{'*' if p in af_set else ''}" for p in pids], fontsize=7)
    ax.set_xlabel("time in recording [s]")
    ax.set_title(f"GT AF timeline (window={WINDOW_SEC}s)  green=AF red=non-AF grey=n/a  (*=clinical AF)")
    plt.tight_layout(); plt.show()

## 3 · Feature extraction (contactless only) + labels

In [ ]:
cfg = X.ExtractConfig(data_root=DATA_ROOT, af_list=AF_LIST, fs=FS,
                      window_s=WINDOW_SEC, hop_s=HOP_SEC, bcg_mode=BCG_MODE, min_valid_hrv=0)
df = X.load_or_extract(cfg, RESULTS_DIR)
print("feature table:", df.shape)
bcg_cols = X.expert_feature_cols(df, "bcg")
leak = [c for c in bcg_cols if any(k in c.lower() for k in ("meanrr","sdnn","cosen","rmssd","acf"))]
print(f"BCG columns: {len(bcg_cols)} | RR/HRV leakage: {leak if leak else 'none'}")

def attach_labels(df, source):
    df = df.copy()
    if source == "patient":
        df["y"] = df["AF"].astype(float)
    elif source == "gt_window":
        gt_lab = {(pid, int(r.win_idx)): r.label
                  for pid, t in gt_tables.items() for r in t.itertuples()}
        df["y"] = [gt_lab.get((p, int(ts // WINDOW_SEC)), np.nan)
                   for p, ts in zip(df["patient"], df["t_start_s"])]
    else:
        raise ValueError(source)
    return df

df = attach_labels(df, LABEL_SOURCE)
lab = df.dropna(subset=["y"]).reset_index(drop=True)
print(f"labelled windows: {len(lab)} | AF={int((lab.y==1).sum())} | non-AF={int((lab.y==0).sum())}")
assert lab["y"].nunique() == 2, "Need two classes — check LABEL_SOURCE / AF_LIST."

## 4 · Three modality experts (LOPO out-of-fold probabilities)

One `HistGradientBoostingClassifier` per modality (handles NaN windows natively; swap freely),
each restricted to its own columns. Every window's probability comes from a model that never saw
that patient — these OOF probabilities feed the gate and every metric below.

In [ ]:
def metrics(y, p, thr=0.5):
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    g = lambda a, b: a / b if b else np.nan
    return dict(acc=(tp+tn)/len(y), sens=g(tp, tp+fn), spec=g(tn, tn+fp),
                ppv=g(tp, tp+fp), npv=g(tn, tn+fn), f1=g(2*tp, 2*tp+fp+fn),
                auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan)

MODALITIES = ["cecg", "ppg", "bcg"]
y = lab["y"].astype(int).values
groups = lab["patient"].values

def expert_oof_probs(data, y, groups):
    P = np.full((len(data), len(MODALITIES)), np.nan)
    for j, mod in enumerate(MODALITIES):
        Xm = data[X.expert_feature_cols(data, mod)].to_numpy(float)
        for tr, te in LeaveOneGroupOut().split(Xm, y, groups):
            if len(np.unique(y[tr])) < 2:
                P[te, j] = y[tr].mean(); continue
            clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
            clf.fit(Xm[tr], y[tr], sample_weight=compute_sample_weight("balanced", y[tr]))
            P[te, j] = clf.predict_proba(Xm[te])[:, 1]
    return P

P = expert_oof_probs(lab, y, groups)
expert_df = pd.DataFrame({m: metrics(y, P[:, j]) for j, m in enumerate(MODALITIES)}).T.round(3)
display(expert_df)

## 5 · SQI gating network

The gate maps per-window **SQI** → softmax over the three experts (how much to trust each
modality this window); final probability `p̂ = Σ wᵢ·pᵢ`, supervised by the labels (BCE).

Two robustness features matter (set in Section 0): a **softmax temperature** and **weight decay**
that bias the gate toward *equal* weights, so in the worst case the gate degrades gracefully to the
naive mean instead of overfitting SQI noise. (SQI alone is ~chance for AF — its only job is
*routing*, not classifying.)

In [ ]:
class GatingNet:
    """SQI -> softmax(weights over experts); blends expert probs. 1 hidden tanh layer, BCE loss.
       Near-uniform init + temperature + weight decay => graceful fallback to the mean."""
    def __init__(self, d_in, n_experts, hidden=GATE_HIDDEN, lr=GATE_LR, epochs=GATE_EPOCHS,
                 temp=GATE_TEMP, wd=GATE_WD, seed=RANDOM_STATE):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1/np.sqrt(d_in), (d_in, hidden)); self.b1 = np.zeros(hidden)
        self.W2 = r.normal(0, 0.01, (hidden, n_experts));        self.b2 = np.zeros(n_experts)
        self.lr, self.epochs, self.temp, self.wd = lr, epochs, temp, wd
        self.mu = self.sd = None

    def _softmax(self, z):
        z = (z / self.temp); z = z - z.max(1, keepdims=True)
        e = np.exp(z); return e / e.sum(1, keepdims=True)

    def weights(self, Xsqi):
        Xs = (Xsqi - self.mu) / self.sd
        return self._softmax(np.tanh(Xs @ self.W1 + self.b1) @ self.W2 + self.b2)

    def fit(self, Xsqi, P, y):
        self.mu, self.sd = Xsqi.mean(0), Xsqi.std(0) + 1e-8
        Xs = (Xsqi - self.mu) / self.sd; y = y.astype(float); eps = 1e-7
        for _ in range(self.epochs):
            h = np.tanh(Xs @ self.W1 + self.b1)
            w = self._softmax(h @ self.W2 + self.b2)
            phat = np.clip((w * P).sum(1), eps, 1 - eps)
            dphat = (phat - y) / (phat * (1 - phat)) / len(y)
            dz2 = (dphat[:, None] * w * (P - phat[:, None])) / self.temp
            dh = (dz2 @ self.W2.T) * (1 - h ** 2)
            self.W2 -= self.lr * (h.T @ dz2 + self.wd * self.W2); self.b2 -= self.lr * dz2.sum(0)
            self.W1 -= self.lr * (Xs.T @ dh + self.wd * self.W1); self.b1 -= self.lr * dh.sum(0)
        return self

    def predict_proba(self, Xsqi, P):
        return (self.weights(Xsqi) * P).sum(1)


sqi_cols = X.gate_sqi_cols(lab, "all")
Xsqi = np.nan_to_num(lab[sqi_cols].to_numpy(float))

gated = np.full(len(lab), np.nan); sqi_only = np.full(len(lab), np.nan)
for tr, te in LeaveOneGroupOut().split(Xsqi, y, groups):
    if len(np.unique(y[tr])) < 2:
        gated[te] = P[tr].mean(); sqi_only[te] = y[tr].mean(); continue
    gated[te] = GatingNet(Xsqi.shape[1], len(MODALITIES)).fit(Xsqi[tr], P[tr], y[tr]) \
                    .predict_proba(Xsqi[te], P[te])
    sqi_only[te] = LogisticRegression(max_iter=2000, class_weight="balanced") \
                       .fit(Xsqi[tr], y[tr]).predict_proba(Xsqi[te])[:, 1]

print(f"SQI-only AUC (sanity, expect ~0.5): {roc_auc_score(y, sqi_only):.3f}")
print(f"Naive mean-of-experts AUC:          {roc_auc_score(y, P.mean(1)):.3f}")
print(f"SQI-GATED experts AUC:              {roc_auc_score(y, gated):.3f}")

## 6 · Results — experts + gating

In [ ]:
rows = {m: metrics(y, P[:, j]) for j, m in enumerate(MODALITIES)}
rows["mean(experts)"] = metrics(y, P.mean(1))
rows["SQI-gated"]     = metrics(y, gated)
results = pd.DataFrame(rows).T.round(3); display(results)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cm = confusion_matrix(y, (gated >= 0.5).astype(int), labels=[0, 1])
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, int(v), ha="center", va="center")
ax[0].set_xticks([0,1]); ax[0].set_xticklabels(["non-AF","AF"]); ax[0].set_yticks([0,1])
ax[0].set_yticklabels(["non-AF","AF"]); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
ax[0].set_title("Gated confusion matrix")
g_all = GatingNet(Xsqi.shape[1], len(MODALITIES)).fit(Xsqi, P, y)
W = g_all.weights(Xsqi)
ax[1].bar(MODALITIES, W.mean(0), yerr=W.std(0), capsize=4, color="#5c6bc0")
ax[1].set_ylabel("mean gate weight"); ax[1].set_title("Average trust per modality")
plt.tight_layout(); plt.show()

## 7 · Per-window inspector — expert guesses + gate weights

For any single window: the three expert AF-probabilities, the gate's weights, the blended
probability, the truth, and the SQI values that drove the weights. Use it to see *why* a window is
right or wrong — e.g. an expert is confidently wrong but the gate failed to down-weight it.

In [ ]:
W_all = g_all.weights(Xsqi)                    # gate weights for every labelled window
gated_all = (W_all * P).sum(1)

def inspect_window(pid=None, win_idx=None, row=None):
    """Inspect one window by (patient, win_idx) or by integer row index into `lab`."""
    if row is None:
        sel = lab.index[(lab.patient == pid) & (lab.win_idx == win_idx)]
        if len(sel) == 0: print("no such window"); return
        row = int(sel[0])
    r = lab.iloc[row]
    print(f"patient={r.patient}  win_idx={int(r.win_idx)}  t_start={r.t_start_s:.0f}s  "
          f"TRUE={'AF' if r.y==1 else 'non-AF'}")
    print(f"  expert probs : " + "  ".join(f"{m}={P[row,j]:.2f}" for j,m in enumerate(MODALITIES)))
    print(f"  gate weights : " + "  ".join(f"{m}={W_all[row,j]:.2f}" for j,m in enumerate(MODALITIES)))
    print(f"  blended p-hat: {gated_all[row]:.2f}  -> predict {'AF' if gated_all[row]>=.5 else 'non-AF'}")
    comp = [c for c in sqi_cols if c.endswith("_composite")]
    if comp:
        print("  SQI(composite): " + "  ".join(f"{c.replace('sqi_','').replace('_composite','')}="
              f"{lab.iloc[row][c]:.2f}" for c in comp))
    fig, ax = plt.subplots(1, 2, figsize=(8, 2.6))
    ax[0].bar(MODALITIES, P[row], color="#26a69a"); ax[0].set_ylim(0,1)
    ax[0].axhline(.5, ls="--", c="grey"); ax[0].set_title("expert P(AF)")
    ax[1].bar(MODALITIES, W_all[row], color="#5c6bc0"); ax[1].set_ylim(0,1)
    ax[1].set_title("gate weights"); plt.tight_layout(); plt.show()

# Example: the most "wrong" window (largest gap between blended prediction and truth)
worst = int(np.argmax(np.abs(gated_all - y)))
print("=== Example: worst-predicted window ===")
inspect_window(row=worst)

def list_disagreements(n=10):
    """Windows where experts disagree most (std of their probs) — prime debugging candidates."""
    d = lab.copy()
    d["expert_std"] = P.std(1); d["blended"] = gated_all; d["truth"] = y
    cols = ["patient","win_idx","truth","blended","expert_std"]
    return d.sort_values("expert_std", ascending=False)[cols].head(n).round(3)

print("\n=== Windows where experts disagree most ===")
display(list_disagreements())
# Inspect any of them, e.g.:  inspect_window("PAT001", 12)

## 8 · Diagnostics — *why* the gate helps (or doesn't), and how to fix it

If the gated AUC is **below** the naive mean, the gate is adding variance without useful routing.
The cells below quantify whether useful routing even exists and try concrete fixes.

In [ ]:
# ── 8a · Routing headroom: is there anything for a gate to exploit? ───────────
mean_auc = roc_auc_score(y, P.mean(1))
conf_toward_truth = np.where(y[:, None] == 1, P, 1 - P)     # how "right" each expert is
oracle = P[np.arange(len(P)), conf_toward_truth.argmax(1)]
oracle_auc = roc_auc_score(y, oracle)
mean_wrong = (P.mean(1) >= .5).astype(int) != y
any_expert_right = (((P >= .5).astype(int) == y[:, None]).any(1))
rescuable = (mean_wrong & any_expert_right).mean()
print(f"naive mean AUC          : {mean_auc:.3f}")
print(f"SQI-gated AUC           : {roc_auc_score(y, gated):.3f}")
print(f"oracle (upper bound) AUC: {oracle_auc:.3f}")
print(f"windows the mean gets wrong but some expert gets right: {rescuable:.1%}")
print("\nReading it: if oracle >> mean, routing CAN help and the gate needs better inputs/training.\n"
      "If oracle ~= mean, the experts are redundant and the simple mean is the right answer —\n"
      "keep the mean and spend effort on the experts, not the gate.")

In [ ]:
# ── 8b · Calibrate expert probabilities, then blend (often helps the ensemble) ─
Pc = np.zeros_like(P)
for j in range(len(MODALITIES)):
    iso = IsotonicRegression(out_of_bounds="clip").fit(P[:, j], y)
    Pc[:, j] = iso.predict(P[:, j])
print(f"mean-of-experts AUC  raw={roc_auc_score(y, P.mean(1)):.3f}  "
      f"calibrated={roc_auc_score(y, Pc.mean(1)):.3f}")
print("(diagnostic: isotonic fit in-sample on OOF probs; for a final number nest it in the LOPO loop)")

In [ ]:
# ── 8c · Patient-level decision (the clinically meaningful number) ────────────
pl = pd.DataFrame({"patient": lab.patient.values, "y": y,
                   "mean": P.mean(1), "gated": gated})
agg = pl.groupby("patient").agg(truth=("y", "max"),
                                mean_score=("mean", "mean"),
                                gated_score=("gated", "mean")).reset_index()
print("Patient-level metrics over", len(agg), "patients (window probs averaged per patient):")
display(pd.DataFrame({
    "mean(experts)": metrics(agg.truth, agg.mean_score),
    "SQI-gated":     metrics(agg.truth, agg.gated_score),
}).T.round(3))
print("\nPatient-level is what the thesis ultimately reports; it smooths per-window noise and is\n"
      "robust to a few mislabelled windows. With 40 patients, treat these CIs as wide.")

## 9 · What to try / change

**GT validation (Section 2)**
- `RPEAK_METHOD` — `'neurokit'`, `'pantompkins1985'`, `'hamilton2002'`. Compare beat counts in the
  overlay plot; pick whatever tracks the ECG best on your hardware.
- `GT_MIN_BEATS`, `GT_HR_RANGE` — the quality gate. Tighten if noisy windows still slip through as
  AF; loosen if too many windows go *n/a*.
- `AF_CV_THR`, `AF_PNN_THR` — how irregular RR must be to call AF. Use the QA scatter/histogram to
  pick a threshold that separates your AF and control patients.
- The QA cell flags patients whose detector AF-fraction contradicts their clinical label — inspect
  those recordings directly (paroxysmal AF, lead noise, or a label issue).

**Labels** — `LABEL_SOURCE='patient'` for your 21-AF / 19-control cohort. `'gt_window'` only if a
cohort is all-AF.

**Experts** — swap `HistGradientBoostingClassifier` for RandomForest/SVM/etc; keep balanced
`sample_weight`. BCG wavelet hyperparameters (`db4`, level 6, PAPR domain) live in `features.py`.

**Gate** — `GATE_TEMP` (up = softer, closer to equal weights), `GATE_WD` (up = stronger pull to the
mean), `GATE_HIDDEN/EPOCHS/LR`. Read Section 8a first: only invest in the gate if the oracle shows
routing headroom. Otherwise the **naive mean is the honest baseline to beat**.

**Inspector (Section 7)** — `inspect_window("PAT0xx", k)` for any window; `list_disagreements()` to
find the informative ones.

**Decision level** — report **patient-level** metrics (Section 8c) as the headline; per-window as a
secondary, finer-grained view.

**PyTorch + Optuna (future)** — replace `GatingNet` with an `nn.Module` of the same shape
(`forward(SQI) -> softmax(3)`, `p̂ = Σ wᵢpᵢ`, BCE). Keep the LOPO loop. Let Optuna tune
hidden size/depth/lr/weight-decay/temperature. Move here once Section 8a shows the gate can beat the
mean — the current results suggest tuning the experts and calibration may matter more than the gate.

**Always keep LOPO.** Windows of one patient are highly correlated; mixing them across train/test
inflates every score.